# Command Line Interfaces (CLI)
<img src='../../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

Command Line Interfaces allow users to interact with your code through the terminal. In this notebook, you will learn three approaches to building CLIs in Python:

1. **Python Scripts** - Basic `.py` files that can be executed
2. **argparse** - Python's built-in argument parsing library
3. **Typer** - A modern CLI library built on top of type hints

## 1. Python Scripts

The simplest way to create a CLI is to write a Python script that can be executed directly. Let's create a simple script that loads data and prints some information about it.

In [ ]:
%%writefile ./simple_script.py
"""A simple script to load and analyze animal shelter data."""

def say_hi():
    print("Hello")

if __name__ == "__main__":
    say_hi()

You can run this script from the terminal:

```bash
python simple_script.py
```

Or from the notebook:

In [ ]:
!python simple_script.py

### Limitations of Simple Scripts

- No way to configure behavior without editing the script
- No help text or documentation for users
- Difficult to reuse or extend

Let's see how we can improve this with `argparse`.

## 2. Using argparse

`argparse` is Python's built-in library for parsing command-line arguments. It allows you to:
- Accept arguments and options from users
- Automatically generate help messages
- Validate input
- Provide default values

In [ ]:
%%writefile ./argparse_cli.py
"""CLI using argparse."""

import argparse

def say_hi(name):
    print(f"Hello {name}")

def main():
    # Create argument parser
    parser = argparse.ArgumentParser(
        description='CLI using argparse',
    )
    
    # Required positional argument
    parser.add_argument(
        'name',
        type=str,
        help='Name'
    )
    
    # Parse arguments
    args = parser.parse_args()
    
    # Call the main function
    say_hi(
        name=args.name,
    )

if __name__ == "__main__":
    main()

Now you can run the script with various options:

```bash
# Show help
python argparse_cli.py --help

# Basic usage
python argparse_cli.py World
```

In [ ]:
# Show help
!python argparse_cli.py --help

In [ ]:
# Run with positional arguments
!python argparse_cli.py World

### argparse Features

- **Positional arguments**: Required arguments (like `name`)
- **Optional arguments**: Start with `--` or `-` (like `--verbose`, `-v`)
- **Type validation**: Automatically converts and validates types
- **Default values**: Arguments can have default values
- **Action types**: `store_true`, `store_false`, `append`, etc.
- **Help text**: Automatically generated from descriptions

### Limitations of argparse

- Verbose and boilerplate-heavy
- Manual type conversion and validation
- No automatic completion
- Error messages can be unclear
- Difficult to test

## 3. Using Typer

Typer is a modern library for building CLIs that uses Python type hints. It's built on top of Click and provides:
- Automatic help text from docstrings
- Type validation from type hints
- Beautiful error messages
- Automatic shell completion
- Easy testing
- Subcommands support

In [ ]:
%%writefile ./typer_cli.py
"""CLI using Typer."""

import typer

app = typer.Typer(
    name="CLI using Typer"
)

@app.command()
def say_hi(name):
    """Say hello to the user."""
    typer.echo(f"Hello: {name}")

@app.command()
def goodbye():
    """Say goodbye to the user."""
    typer.echo("Goodbye!")

if __name__ == "__main__":
    app()

Typer provides a much cleaner syntax and powerful features:

```bash
# Show help
python typer_cli.py --help

# Show help for a specific command
python typer_cli.py say-hi --help

# Analyze command
python typer_cli.py say-hi World
```

In [ ]:
# Show main help
!python typer_cli.py --help

In [ ]:
# Show help for say-hi command
!python typer_cli.py say-hi --help

In [ ]:
# Run say-hi command
!python typer_cli.py say-hi World

## <mark>Exercise 1:</mark> Add Another Command

Add a new command to the Typer CLI called `greet` that:
1. Takes a `name` as a required argument
2. Prints a greeting message to that person

Your command should work like this:
```bash
python typer_cli.py greet Alice
# Output: Greetings, Alice!
```

## <mark>Exercise 2:</mark> Add a Command with Optional Arguments

Add a new command to the Typer CLI called `welcome` that:
1. Takes a `name` as a required argument
2. Has an optional `--greeting` option that allows custom greeting text (default: "Hello")
3. Has an optional `--enthusiastic` flag that adds exclamation marks when enabled

Your command should work like this:
```bash
python typer_cli.py welcome Alice
# Output: Hello Alice

python typer_cli.py welcome Alice --greeting "Good morning"
# Output: Good morning Alice

python typer_cli.py welcome Alice --enthusiastic
# Output: Hello Alice!!!
```

## Typer Features

### Commands and Subcommands
- Use `@app.command()` to create commands
- Each function becomes a subcommand
- Docstrings become help text

### Type Hints
Typer uses Python type hints to:
- Automatically validate input types
- Convert arguments to the correct type
- Generate better help text

### Annotated Parameters
The `Annotated` type (from `typing_extensions`) allows you to add metadata to parameters:
```python
name: Annotated[str, typer.Argument(help="Name of a creature to say hi to")]
verbose: Annotated[bool, typer.Option("--verbose", "-v", help="Enable verbose mode")]
```

### Options vs Arguments
- **Arguments**: Required positional parameters (e.g., `name`)
- **Options**: Optional named parameters with `--` prefix (e.g., `--verbose`)

### Boolean Flags
- `--flag/--no-flag` creates a boolean toggle
- `--verbose` with `action='store_true'` creates a simple flag

## Comparison Summary

| Feature | Simple Script | argparse | Typer |
|---------|--------------|----------|-------|
| Ease of use | Very easy | Moderate | Easy |
| Configuration | None | Manual | Automatic |
| Type validation | None | Manual | Automatic |
| Help text | None | Manual | From docstrings |
| Subcommands | No | Difficult | Easy |
| Testing | Difficult | Moderate | Easy |
| Shell completion | No | No | Yes |
| Code verbosity | Low | High | Low |

### When to use each:

- **Simple Script**: Quick one-off tasks, internal use only
- **argparse**: Standard library solution, no external dependencies needed
- **Typer**: Modern CLIs, better UX, easier maintenance

## Using the CLI Package

Registering CLIs in `uv`

> The code that uses your application should be separate from the code that implements it.

→ **Your CLI / API code should be implemented in a different package than `animal_ shelter`**.

* In `uv`, this is achieved by defining your CLI/API as a special package called a `workspace member`.
* Workspace members are separate packages that live in your main project directory and are synced to your project's main `pyproject.toml`.
* However, they also have their own `pyproject.toml` which allows for flexibility in defining dependencies or their own registered scripts.
* It is common to put all workspace members into a common subdirectory within the project (e.g., packages/*)

The `packages/cli` package in this repository provides a complete CLI application. Let's explore it:

First, you need to create a `packages` folder under the root project.

Underneath this `packages` folder, initialize the CLI project:

```shell
uv init cli --package --name cli --no-readme
```

Now, you can see `uv` create a package `cli` with `pyproject.toml` and `src` folder.

In [ ]:
!ls ../../packages/cli

In [ ]:
%%writefile ../../packages/cli/src/cli/cli.py
"""CLI using Typer."""

import typer

app = typer.Typer(
    name="CLI using Typer"
)

@app.command()
def say_hi(name):
    """Say hello to the user."""
    typer.echo(f"Hello: {name}")

@app.command()
def goodbye():
    """Say goodbye to the user."""
    typer.echo("Goodbye!")

if __name__ == "__main__":
    app()

### Update CLI's `pyproject.toml`

1. Add `typer` in the `dependencies`;
2. Update `[project.scripts]` to point to the right Typer app. 

```toml
dependencies = [
    "typer>=0.25.0",
]

[project.scripts]
cli = "cli.cli:app"
```

### Update Animal Shelter's `pyproject.toml`

```toml
dependencies = [
    "pandas>=2.3.3",
    "fastapi>=0.115.0",
    "uvicorn>=0.34.0",
    "scikit-learn>=1.8.0",
    "pydantic>=2.10.0",
    "prometheus-client>=0.21.0",
    "cli",
]

[tool.uv.workspace]
members = [
    "packages/cli",
]

[tool.uv.sources]
cli = { workspace = true }
```

In [ ]:
!uv run cli --help

## <mark>Exercise 3:</mark> Extend the CLI Package

Modify the CLI package in `packages/cli/src/cli/cli.py` to:

1. Add a `@app.callback()` that prints a welcome message
2. Test your changes by running the CLI

**Hints:** You can check out https://typer.tiangolo.com/tutorial/commands/callback/ to see how to implement `app.callback`.

## Key Takeaways

1. **Start simple**: Begin with a basic script, add argparse when you need options, use Typer for production CLIs
2. **Type hints are powerful**: Typer leverages type hints for validation and better UX
3. **Docstrings matter**: They become your help text in Typer
4. **Separate logic from CLI**: Keep business logic in separate functions, use CLI just for the interface
5. **Test your CLI**: Typer makes it easy to test CLI applications programmatically